In [ ]:
# Match_Extract_Soldier_Affiliation.py
# 6/30/2025


from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance

from dateutil import parser
from dateutil.relativedelta import relativedelta
import ast

def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

    
def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC','TEC5']
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df     

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df

def match_sets(soldier_match_list_df,search,column_search): 
    search_set= ast.literal_eval(search)
    if isinstance(soldier_match_list_df[column_search].iloc[0], set)==False:
        soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set()) 
    soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
        lambda x: {'nan'} if pd.isna(x) else x)
    soldier_match_list_df = soldier_match_list_df.copy()
    soldier_match_list_df.loc[:,'set_difference']=search_set-soldier_match_list_df[column_search]
    return soldier_match_list_df

def discard_set_one_character(data_set_df,column):
    data_set_df=data_set_df.copy()
    data_set_df[column] = data_set_df[column].apply(
        lambda x: {item for item in ast.literal_eval(x) if len(item) > 2} if isinstance(x, str) else 
              ({item for item in x if len(item) > 1} if isinstance(x, set) else x))
    return data_set_df

def drop_sets_datafrme_duplicates(dataframe_set_df):
    columns_set = [col for col in soldier_child_match_df.columns if '_set' in col]
    dataframe_set_df=soldier_child_match_df
    
    # set_columns = ['col1', 'col2']  # replace with your actual set column names
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df




maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Match and Extract Soldier Affiliation'
print('**Starting***',process_name,'***,Time:',start_time,'**\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

grave_source='ABMC'
letter='A'
#grave_source='Arlington'

#relative_list=['Parents','Siblings','Spouses','Children']


fileInput=grave_source+"_Soldier_Parent_Merge_"+letter
fileOutput=grave_source+"_Soldier Affiliation_"+letter

fileIn=DirMain+"Analysis\\"+fileInput+".csv"
fileOut=DirMain+"Analysis\\"+fileOutput+".csv"
fileOutFail=DirMain+"Analysis\\"+fileOutput+"_failed.csv"
fileOutTest1=DirMain+"Analysis\\"+fileOutput+"_Test1.csv"

soldier_parent_df=pd.read_csv(fileIn)

soldier_parent_df['cemetery_id']=soldier_parent_df['cemetery_id'].astype(int)



field_list=['soldier_name_index', 'soldier_last_name', 'soldier_first_name', 'soldier_middle_name', 'soldier_full_name',\
                     'soldier_death_index', 'entry_state', 'soldier_birth_state', 'parents_name_index',\
                     'parents_name_full', 'parents_name_full_set', 'soldiers_full_name_set', 'age_at_birth', 'soldier_abmc_missing',\
                     'match_name_index_short', 'match_full_name', 'match_full_name_set', 'spouses_set',\
                     'parents_birth_state', 'parents_birth_country', 'category', 'cemetery_name', 'cemetery_id',\
                     'url', 'spouses', 'children', 'children_set', 'children_set_of_sets']

soldier_parent_work_df=soldier_parent_df[['soldier_name_index', 'soldiers_full_name', 'soldiers_full_name_set','parents_name_full','parents_name_full_set','match_full_name_set','spouses','spouses_set','children_set',\
                          'category', 'cemetery_name', 'cemetery_id','entry_state','soldier_birth_state','parents_birth_country', 'children_set']]

#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
soldiers_name_list_df=pd.DataFrame(soldier_parent_work_df['soldiers_full_name_set'].unique())
soldiers_name_list_df.columns=['soldiers_name_list']

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

column_search='spouses_set'
soldier_match_df=pd.DataFrame([])
soldier_match_df_failed=pd.DataFrame([])
for index1, row1 in soldiers_name_list_df.iterrows():
    soldiers_name_full_set=row1['soldiers_name_list']
    soldier_match_list_df=soldier_parent_work_df[soldier_parent_work_df['soldiers_full_name_set']==soldiers_name_full_set]
    row_count=0
    for  index2, row2 in soldier_match_list_df.iterrows():
        soldier_full_name=row2['soldiers_full_name']
        parent_spouse_match_df=pd.DataFrame([])
        parent_name=row2['parents_name_full_set']
        parent_full_name=row2['parents_name_full']
        spouse_name=row2['spouses_set']
        spouse_full_name=row2['spouses']
        # print(f" ** Soldier: {soldier_full_name} • Parent: {parent_full_name} • Spouse: {spouse_full_name}")
        column_search='parents_name_full_set'
        search=spouse_name
        
        soldier_match_list_df= match_sets(soldier_match_list_df,search,column_search)
        parent_spouse_match_df=soldier_match_list_df[soldier_match_list_df['set_difference']==set()]
        if len(parent_spouse_match_df)==0: pd.concat([soldier_match_df_failed,soldier_match_list_df])
        parent_spouse_match_df=parent_spouse_match_df.copy()
        
        parent_spouse_match_df['spouse_matched_parent']=parent_full_name
 
        
        parent_spouse_match_df_select=parent_spouse_match_df[['set_difference','soldier_name_index','soldiers_full_name_set','spouse_matched_parent','parents_name_full',\
                                'category','cemetery_name','cemetery_id']]
        parent_spouse_match_df_select.columns=['set_difference','soldier_name_index','soldiers_name_set','parent','spouse',\
                                'category','cemetery_name','cemetery_id']

        # soldier_match_name_df=pd.DataFrame(parent_spouse_match_df_select).T        
        soldier_match_df=pd.concat([soldier_match_df,parent_spouse_match_df_select])
soldier_match_df.to_csv(fileOut,mode="w",index=False,header=True)
soldier_match_df_failed.to_csv(fileOutFail,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut)
        


In [330]:
# ******************** children_set pset processing **********************
def drop_sets_datafrme_duplicates(dataframe_set_df):
    columns_set = [col for col in soldier_child_match_df.columns if '_set' in col]
    dataframe_set_df=soldier_child_match_df
    
    # set_columns = ['col1', 'col2']  # replace with your actual set column names
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df


maximize_screen()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

soldier_child_match_df=pd.DataFrame([])
soldier_parent_work_df=soldier_parent_df[['soldier_name_index', 'soldiers_full_name', 'soldiers_full_name_set','parents_name_full','parents_name_full_set','match_full_name_set','spouses','spouses_set','children_set',\
                          'category', 'cemetery_name', 'cemetery_id','entry_state','soldier_birth_state','parents_birth_country']]
soldier_parent_work_df1 = soldier_parent_work_df.iloc[0:1100]
soldier_parent_work_df2 = soldier_parent_work_df.iloc[0:1100]
column='soldiers_full_name_set'
soldier_parent_work_df1=discard_set_one_character(soldier_parent_work_df1,column)

column_s='children_set'
soldier_parent_work_df1=soldier_parent_work_df1.copy()
soldier_parent_work_df1['children_set'] = soldier_parent_work_df1['children_set'].apply(ast.literal_eval)
soldier_parent_work_exploded_df1 = soldier_parent_work_df1.explode('children_set')
soldier_parent_work_exploded_df1=clean_dataframe(soldier_parent_work_exploded_df1,column_s)
soldiers_df_split=soldier_parent_work_exploded_df1['children_set'].str.split(' ')
children_names=soldiers_df_split.apply(set)
for irow in range(len(soldier_parent_work_df1)):
    if isinstance(children_names[irow] == {'nan'}, bool):    
        soldier_parent_work_df1.loc[irow,'child_match']='No Children Match'
        soldier_child_row_match_df=pd.DataFrame(soldier_parent_work_df1.loc[irow]).T
        soldier_child_match_df=pd.concat([soldier_child_match_df,soldier_child_row_match_df])        
        continue
    soldiers_full_name=soldier_parent_work_df1['soldiers_full_name'].iloc[irow]
    soldiers_full_name_set=soldier_parent_work_df1['soldiers_full_name_set'].iloc[irow]
    children_missing_set=soldiers_full_name_set-children_names[irow]
    # print(soldiers_full_name_set)
    # print(children_missing_set)
    # print('\n')
    soldier_parent_work_df1.loc[irow,'child_match']='No Children Match'
    if (children_missing_set==set()).sum()>=1:soldier_parent_work_df1.loc[irow,'child_match']=soldiers_full_name
    soldier_child_row_match_df=pd.DataFrame(soldier_parent_work_df1.loc[irow]).T
    soldier_child_match_df=pd.concat([soldier_child_match_df,soldier_child_row_match_df])
    # print(soldier_child_match_df)
    # print('\n')
    # print(soldiers_full_name_set)
    # print(children_missing_set)
    # input('')
soldier_child_match_df=drop_sets_datafrme_duplicates(soldier_child_match_df)
fileOutChildren=DirMain+"Analysis\\"+fileOutput+"_Children.csv"
soldier_child_match_df.to_csv(fileOutChildren,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOutChildren,'Number of rows:',len(soldier_child_match_df))
print('\n',soldier_parent_work_df1['child_match'])



UploadedTo: C:\Users\jrozm\OneDrive\Benjamin\Analysis\ABMC_Soldier Affiliation_A_Children.csv Number of rows: 1091

 0       George Edward Aaron
1       George Edward Aaron
2         John Daniel Aaron
3         John Daniel Aaron
4         No Children Match
               ...         
1095      No Children Match
1096      No Children Match
1097      No Children Match
1098      No Children Match
1099           Howard Adams
Name: child_match, Length: 1100, dtype: object


In [329]:
# Convert set columns to sorted tuples
columns_set = [col for col in soldier_child_match_df.columns if '_set' in col]
dataframe_set_df=soldier_child_match_df

# set_columns = ['col1', 'col2']  # replace with your actual set column names
for col in set_columns:
    dataframe_set_df[col] = dataframe_set_df[col].apply(
        lambda x: tuple(sorted(x)) if isinstance(x, set) else x)

dataframe_set_df = dataframe_set_df.drop_duplicates()
dataframe_set_df=dataframe_set_df.copy()
for col in set_columns:
    dataframe_set_df[col] = dataframe_set_df[col].apply(
        lambda x: set(x) if isinstance(x, tuple) else x )
dataframe_set_df

# soldier_child_columns_df.str.contains('_set')
# soldier_child_columns_df




,soldier_name_index,soldiers_full_name,soldiers_full_name_set,parents_name_full,parents_name_full_set,match_full_name_set,spouses,spouses_set,children_set,category,cemetery_name,cemetery_id,entry_state,soldier_birth_state,parents_birth_country,child_match
0,Aaron-George-19441217,George Edward Aaron,"{George, Aaron, Edward}",James Burke Aaron,"{'Burke', 'James', 'Aaron'}","{'Burke', 'James', 'Aaron'}",Sarah T Hollobaugh Aaron,"{'Aaron', 'Hollobaugh', 'Sarah', 'T'}","{Dorothy R Aaron Baughman, George Edward Aaron...",C,Mount Airy Cemetery,45471,Pennsylvania,Pennsylvania,United States of America,George Edward Aaron
1,Aaron-George-19441217,George Edward Aaron,"{George, Aaron, Edward}",Sarah T Hollobaugh Aaron,"{'Aaron', 'Hollobaugh', 'Sarah', 'T'}","{'Aaron', 'Hollobaugh', 'Sarah', 'T'}",James Burke Aaron,"{'Burke', 'James', 'Aaron'}","{Dorothy R Aaron Baughman, George Edward Aaron...",C,Mount Airy Cemetery,45471,Pennsylvania,Pennsylvania,NaN,George Edward Aaron
2,Aaron-John-19441127,John Daniel Aaron,"{John, Daniel, Aaron}",Margaret Adaline Addie Lee Aaron,"{'Addie', 'Margaret', 'Aaron', 'Lee', 'Adaline'}","{'Addie', 'Margaret', 'Aaron', 'Lee', 'Adaline'}",Russell Preston Aaron,"{'Aaron', 'Russell', 'Preston'}","{Susan Mae Sue Aaron Perry, Leota Aaron Stone,...",C,Magnolia Memorial Cemetery,98875,Oklahoma,Arkansas,United States of America,John Daniel Aaron
3,Aaron-John-19441127,John Daniel Aaron,"{John, Daniel, Aaron}",Russell Preston Aaron,"{'Aaron', 'Russell', 'Preston'}","{'Aaron', 'Russell', 'Preston'}",Margaret Adaline Addie Lee Aaron * Fanny Bell...,"{'*', 'Addie', 'Margaret', 'Barnard', 'Aaron',...","{Susan Mae Sue Aaron Perry, Leota Aaron Stone,...",C,Magnolia Memorial Cemetery,98875,Oklahoma,Arkansas,United States of America,John Daniel Aaron
4,Aaron-John-19441127,John Daniel Aaron,"{John, Daniel, Aaron}",Russell Preston Aaron,"{'Aaron', 'Russell', 'Preston'}","{'Aaron', 'Russell', 'Preston'}",NaN,{'nan'},{nan},C,Rattan Cemetery,99326,Oklahoma,Arkansas,United States of America,No Children Match
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,Adams-Howard-19421126,Howard Adams,"{Howard, Adams}",Annie M Adams,"{'Adams', 'Annie', 'M'}","{'Adams', 'Annie', 'M', 'Haws'}",Martin Adams,"{'Adams', 'Martin'}","{Margaret I Maggie Adams Hallman, Hettie M Ada...",C,Salem Union Cemetery,46285,Arkansas,Arkansas,United States of America,No Children Match
1096,Adams-Howard-19421126,Howard Adams,"{Howard, Adams}",Annie M Adams,"{'Adams', 'Annie', 'M'}","{'Adams', 'Annie', 'Hill', 'M'}",Novie Holice Adams,"{'Adams', 'Novie', 'Holice'}",{Novie Adams},C,South Pleasant Grove Cemetery,534915,Arkansas,Arkansas,NaN,No Children Match
1097,Adams-Howard-19421126,Howard Adams,"{Howard, Adams}",Annie M Adams,"{'Adams', 'Annie', 'M'}","{'Adams', 'Annie', 'M'}",Sam Houston Adams,"{'Sam', 'Adams', 'Houston'}",{nan},N,Chattanooga National Cemetery,109392,Arkansas,Arkansas,NaN,No Children Match
1098,Adams-Howard-19421126,Howard Adams,"{Howard, Adams}",Annie M Adams,"{'Adams', 'Annie', 'M'}","{'Adams', 'Annie', 'M'}",William H Adams,"{'Adams', 'William', 'H'}",{nan},C,Chattanooga Memorial Park,10429,Arkansas,Arkansas,NaN,No Children Match


In [ ]:
fileOutTest1=DirMain+"Analysis\\"+fileOutput+"_Test1.csv"
soldier_parent_work_df=soldier_parent_df[['soldier_name_index', 'soldiers_full_name', 'soldiers_full_name_set','parents_name_full','parents_name_full_set','match_full_name_set','spouses','spouses_set','children_set',\
                          'category', 'cemetery_name', 'cemetery_id','entry_state','soldier_birth_state','parents_birth_country', 'children_set']]


field_list_results=['soldier_name_index_x', 'soldiers_full_name_x', 'parents_name_full_x', 'spouses_x', 'parents_name_full_y',\
         'spouses_set_x', 'parents_name_full_set_y', 'category_x', 'cemetery_name_x', 'cemetery_id_x','children_set_x', 'soldiers_full_name_set_x']
field_list_results_rename=['soldier_index', 'soldiers_full_name', 'parents_full_name', 'spouses_full_name', 'matched_spouses_full_name',\
         'spouses_set_x', 'parents_name_full_set_y', 'category', 'cemetery_name', 'cemetery_id','children_set', 'soldiers_full_name_set']

 

# Make a copy for the self-join
soldier_parent_work_df1 = soldier_parent_work_df.iloc[0:1100]
soldier_parent_work_df2 = soldier_parent_work_df.iloc[0:1100]

first_join_df=pd.merge(
soldier_parent_work_df1,
soldier_parent_work_df2,
left_on='spouses_set',
right_on='match_full_name_set',
how='left'#or'left','inner','right','outer'dependingonwhatyouwant
)
first_join_df_out=first_join_df[field_list_results]
first_join_df_out.columns=field_list_results_rename
print('Before:',len(first_join_df_out))
first_join_df_out=first_join_df_out.copy()
first_join_df_out.drop_duplicates(inplace=True)

print('After:',len(first_join_df_out))


print(first_join_df_out)
first_join_df_out.to_csv(fileOutTest1,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOutTest1)
input('')

# Rename columns on second copy to distinguish after join
soldier_parent_work_df2 = soldier_parent_work_df2.rename(columns={
    'parents_name_full_set': 'match_parents_name_set',
    'spouses_set': 'match_spouse_set'})

# Cross join to compare all rows
merged = soldier_parent_work_df1.merge(soldier_parent_work_df2, how='cross')

# Filter where df1['spouse'] contains df2['match_parent']
result = merged[merged.apply(lambda row: row['match_parents_name_set'] in row['spouses_set'], axis=1)]
result_out=['soldier_name_index_x',
 'parents_name_full_set',
 'match_full_name_set_x',
 'spouses_set',
 'children_set_x',
 'entry_state_x',
 'soldier_birth_state_x',
 'parents_birth_state_x',
 'soldier_name_index_y',
 'match_parents_name_set',
 'match_full_name_set_y',
 'match_spouse_set',
 'children_set_y',
 'entry_state_y',
 'soldier_birth_state_y',
 'parents_birth_state_y']

# Optional: drop rows where it's a self-match
# result = result[result['person_id'] != result['match_id']]

print(result)
result.to_csv(fileOut,mode="w",index=False,header=True)
print("\n")
print('UploadedTo:',fileOut)

In [ ]:
soldier_parent_work_df=soldier_parent_df[['soldier_name_index','parents_name_full_set','match_full_name_set','spouses_set','children_set',\
                         'entry_state','soldier_birth_state','parents_birth_state']]

soldier_parent_work_df.loc[:,'parents_name_full_set'] = soldier_parent_work_df['parents_name_full_set'].apply(ast.literal_eval)
soldier_parent_work_df.loc[:,'match_full_name_set'] = soldier_parent_work_df['match_full_name_set'].apply(ast.literal_eval)
soldier_parent_work_df.loc[:,'spouses_set'] = soldier_parent_work_df['spouses_set'].apply(ast.literal_eval)

soldier_parent_work_df.loc[:,'spouses_set'] = soldier_parent_work_df['spouses_set'].apply(
    lambda x: {'nan'} if pd.isna(x) else x)
soldier_parent_work_df.loc[:,'match_full_name_set'] = soldier_parent_work_df['match_full_name_set'].apply(
    lambda x: {'nan'} if pd.isna(x) else x)
soldier_parent_work_df.loc[:,'set_difference']=soldier_parent_work_df['match_full_name_set'].iloc[0]-soldier_parent_work_df['spouses_set']
soldier_parent_match_df=soldier_parent_work_df[soldier_parent_work_df['set_difference']==set()]
soldier_parent_match_df
# soldier_parent_work_df['spouses_set']
# soldier_parent_work_df['match_full_name_set']

In [ ]:
soldier_parent_work_df=soldier_parent_df[['soldier_name_index','parents_name_full_set','match_full_name_set','spouses_set','children_set',\
                         'entry_state','soldier_birth_state','parents_birth_state']]

search=soldier_parent_work_df['match_full_name_set'].iloc[0]
search_set= ast.literal_eval(search)

column_match='spouses_set'
soldier_parent_work_df.loc[:,column_match] = soldier_parent_work_df[column_match].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set())

soldier_parent_work_df = soldier_parent_work_df.copy()
soldier_parent_work_df.loc[:,column_match] = soldier_parent_work_df[column_match].apply(
lambda x: {'nan'} if pd.isna(x) else x)

soldier_parent_work_df.loc[:,'set_difference']=set_search-soldier_parent_work_df['spouses_set']
soldier_parent_match_df=soldier_parent_work_df[soldier_parent_work_df['set_difference']==set()]
soldier_parent_match_df

In [ ]:


soldier_parent_work_df['spouses_set'] = soldier_parent_work_df['spouses_set'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set())
soldier_parent_work_df['match_full_name_set'] = soldier_parent_work_df['match_full_name_set'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set())
soldier_parent_work_df['set_difference']=soldier_parent_work_df['match_full_name_set']-soldier_parent_work_df['spouses_set']
# soldier_parent_work_df.loc[:, 'set_difference'] = soldier_parent_work_df.apply(
#     lambda row: row['match_full_name_set'] - row['spouses_set'] 
#     if isinstance(row['match_full_name_set'], set) and isinstance(row['spouses_set'], set) 
#     else set(),
#     axis=1
# )
soldier_parent_work_df[['match_full_name_set','spouses_set','set_difference']]
# soldier_parent_work_df['spouses_set']

In [ ]:
soldier_parent_df=pd.read_csv(fileIn)
soldier_parent_df['children_set'] = soldier_parent_df['children_set'].apply(ast.literal_eval)
soldier_parent_exploded_df = soldier_parent_df.explode('children_set')
soldier_parent_exploded_df=clean_dataframe(soldier_parent_exploded_df,'children_set')
soldier_parent_exploded_df['children_set']